In [ ]:
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass
from functools import lru_cache
import argparse
import csv
import gzip
import json
import math
import os
import pickle
import random
import sys
import time
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch import nn
from torch.nn.functional import scaled_dot_product_attention
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.distributed import DistributedSampler
from tqdm.auto import tqdm
from transformers import AutoTokenizer


def parse_runtime_overrides() -> Dict[str, Optional[str]]:
    if "ipykernel" in sys.modules:
        return {}
    parser = argparse.ArgumentParser(description="Train the vanilla Transformer reranker with plain PyTorch.")
    parser.add_argument("--profile", choices=["smoke", "full", "full_all_dev"], default=None)
    parser.add_argument("--output-root", default=None)
    parser.add_argument("--resume-from-checkpoint", default=None)
    parser.add_argument("--run-id", default=None)
    args, _ = parser.parse_known_args()
    return {key: value for key, value in vars(args).items() if value is not None}


RUNTIME_OVERRIDES = parse_runtime_overrides()


INPUT_ARTIFACT_DIR = "<path/to/artifacts/trm_reranker_mvp>"
# Path to the prepared artifact directory.
INPUT_PREP_MANIFEST_PATH = "<path/to/prep_manifest.json>"
# Path to prep_manifest.json.
INPUT_RUN_DATA_MANIFEST_PATH = "<path/to/run_data_manifest.json>"
# Path to run_data_manifest.json produced by run-data cache preparation.
INPUT_DATA_ROOT = "<path/to/msmarco_data_dir>"
# Path to the directory containing MS MARCO TSV files.
INPUT_COLLECTION_PATH = "<path/to/collection.tsv>"
# Path to collection.tsv.
ABLATION_OUTPUT_DIRNAME = "vanilla_transformer_rerank"
IS_ABLATION_NOTEBOOK = True
FREEZE_TOKEN_EMBEDDINGS = False
FREEZE_SEGMENT_EMBEDDINGS = False
FREEZE_POSITIONAL_COMPONENTS = False

DEFAULT_OUTPUT_ROOT = RUNTIME_OVERRIDES.get("output_root") or "<path/to/output_dir>"
# Directory where logs, checkpoints, and evaluation files are written.

MANUAL_ARTIFACT_DIR = INPUT_ARTIFACT_DIR
MANUAL_PREP_MANIFEST_PATH = INPUT_PREP_MANIFEST_PATH or (f"{INPUT_ARTIFACT_DIR}/prep_manifest.json" if INPUT_ARTIFACT_DIR else None)
MANUAL_DATA_ROOTS = [INPUT_DATA_ROOT] if INPUT_DATA_ROOT else []
MANUAL_COLLECTION_PATH = INPUT_COLLECTION_PATH or (f"{INPUT_DATA_ROOT}/collection.tsv" if INPUT_DATA_ROOT else None)

RUN_PROFILE = "full"
PROFILE_SETTINGS = {
    "smoke": {
        "train_triples_sample": 100_000,
        "epochs": 1,
        "max_train_steps": 20,
        "run_final_full_dev": True,
        "dev_eval_mode": "quick_fraction",
        "dev_eval_fraction": 0.25,
        "dev_eval_seed": 13,
        "devices": 1,
        "grad_accum_steps": 1,
        "use_ddp": False,
    },
    "full": {
        "train_triples_sample": 20_000_000,
        "epochs": 1,
        "max_train_steps": None,
        "run_final_full_dev": True,
        "dev_eval_mode": "quick_fraction",
        "dev_eval_fraction": 0.1,
        "dev_eval_seed": 13,
        "devices": 2,
        "grad_accum_steps": 1,
        "use_ddp": True,
    },
    "full_all_dev": {
        "train_triples_sample": 1_000_000,
        "epochs": 5,
        "max_train_steps": None,
        "run_final_full_dev": True,
        "dev_eval_mode": "full",
        "dev_eval_fraction": 1.0,
        "dev_eval_seed": 13,
        "devices": 2,
        "grad_accum_steps": 1,
        "use_ddp": True,
    },
}
if RUN_PROFILE not in PROFILE_SETTINGS:
    raise ValueError(f"Unsupported RUN_PROFILE={RUN_PROFILE!r}. Expected one of {sorted(PROFILE_SETTINGS)}")
PROFILE = dict(PROFILE_SETTINGS[RUN_PROFILE])

SEED = 13
TRAIN_TRIPLES_SAMPLE = int(PROFILE["train_triples_sample"])
EPOCHS = int(PROFILE["epochs"])
MAX_TRAIN_STEPS = PROFILE["max_train_steps"]
DEV_EVAL_MODE = str(PROFILE.get("dev_eval_mode", "quick_fraction"))
DEV_EVAL_FRACTION = float(PROFILE.get("dev_eval_fraction", 1.0))
DEV_EVAL_SEED = int(PROFILE.get("dev_eval_seed", SEED))
RUN_FINAL_FULL_DEV = bool(PROFILE.get("run_final_full_dev", True))
DEVICES = int(PROFILE["devices"])
GRAD_ACCUM_STEPS = int(PROFILE["grad_accum_steps"])
USE_DDP = bool(PROFILE["use_ddp"]) and DEVICES > 1
if GRAD_ACCUM_STEPS <= 0:
    raise ValueError("GRAD_ACCUM_STEPS must be >= 1")
if DEV_EVAL_MODE not in {"quick_fraction", "quick_artifact", "full"}:
    raise ValueError(f"Unsupported DEV_EVAL_MODE={DEV_EVAL_MODE!r}")
if not RUN_FINAL_FULL_DEV:
    raise ValueError("RUN_FINAL_FULL_DEV must remain True because final full dev validation is mandatory.")

PER_DEVICE_BATCH_SIZE = 256
EVAL_BATCH_SIZE = 256
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.06
PRECISION = "bf16-mixed"
LOG_EVERY_N_STEPS = 100
NUM_WORKERS = 4
EXPERIMENT_NAME_PREFIX = "vanilla_transformer_rerank"
ALLOW_PREP_MANIFEST_MISMATCH = False
RESUME_FROM_CHECKPOINT = False
RESUME_CHECKPOINT_PATH = ""
if os.environ.get("TRM_RESUME_FROM_CHECKPOINT", "").strip().lower() in {"1", "true", "yes"}:
    RESUME_FROM_CHECKPOINT = True
RESUME_CHECKPOINT_PATH = os.environ.get("TRM_RESUME_CHECKPOINT_PATH") or RUNTIME_OVERRIDES.get("resume_from_checkpoint") or RESUME_CHECKPOINT_PATH


HIDDEN_SIZE = 512
NUM_HEADS = 8
VANILLA_TRANSFORMER_LAYERS = 2
HALT_MAX_STEPS = 1
POS_ENCODINGS = "rope"
EXPANSION = 4.0
TOKENIZER_NAME = "bert-base-uncased"
SEQ_LEN = 256
MAX_QUERY_LEN = 32
MAX_DOC_LEN = 221

REPO_ROOT = Path.cwd().resolve()
OUTPUT_ROOT = Path(DEFAULT_OUTPUT_ROOT).expanduser().resolve()


def sanitize_tag_fragment(value: str) -> str:
    return "".join(ch if ch.isalnum() or ch in "._-" else "-" for ch in value).strip("-") or "tokenizer"


def build_cache_tag(tokenizer_name: str, seq_len: int, max_query_len: int, max_doc_len: int) -> str:
    tokenizer_tag = sanitize_tag_fragment(tokenizer_name)
    return f"{tokenizer_tag}_seq{seq_len}_q{max_query_len}_d{max_doc_len}"


EXPECTED_CACHE_TAG = build_cache_tag(TOKENIZER_NAME, SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN)
RUN_ID = (
    os.environ.get("TRM_RUN_ID")
    or RUNTIME_OVERRIDES.get("run_id")
    or os.environ.get("TORCHELASTIC_RUN_ID")
    or time.strftime("%Y%m%d-%H%M%S")
)
EXPERIMENT_NAME = f"{EXPERIMENT_NAME_PREFIX}_{RUN_PROFILE}_{RUN_ID}"


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_pickle(path: Path):
    with path.open('rb') as handle:
        return pickle.load(handle)


def open_text_auto(path: Path):
    if path.suffix == '.gz':
        return gzip.open(path, 'rt', encoding='utf-8', errors='replace', newline='')
    return path.open('r', encoding='utf-8', errors='replace', newline='')


def get_manifest_mapping(manifest: dict, section_name: str) -> Dict[str, object]:
    value = manifest.get(section_name) or {}
    if not isinstance(value, dict):
        return {}
    return value


def manifest_value_candidates(primary_key: str, aliases: Optional[List[str]] = None) -> List[str]:
    keys = [primary_key]
    if aliases:
        keys.extend(aliases)
    unique: List[str] = []
    seen = set()
    for key in keys:
        if not key or key in seen:
            continue
        seen.add(key)
        unique.append(key)
    return unique


def get_manifest_mapping_value(mapping: Dict[str, object], keys: List[str]):
    for key in keys:
        value = mapping.get(key)
        if value:
            return value, key
    return None, None


def is_dist_available_and_initialized() -> bool:
    return dist.is_available() and dist.is_initialized()


def get_rank() -> int:
    if is_dist_available_and_initialized():
        return dist.get_rank()
    return int(os.environ.get("RANK", "0"))


def get_local_rank() -> int:
    return int(os.environ.get("LOCAL_RANK", "0"))


def get_world_size() -> int:
    if is_dist_available_and_initialized():
        return dist.get_world_size()
    if USE_DDP and int(os.environ.get("WORLD_SIZE", "1")) > 1:
        return int(os.environ.get("WORLD_SIZE", "1"))
    return 1


def is_main_process() -> bool:
    return get_rank() == 0


def ddp_barrier() -> None:
    if is_dist_available_and_initialized():
        dist.barrier()


def reduce_mean(tensor: torch.Tensor) -> torch.Tensor:
    if is_dist_available_and_initialized():
        reduced = tensor.detach().clone()
        dist.all_reduce(reduced, op=dist.ReduceOp.SUM)
        reduced /= dist.get_world_size()
        return reduced
    return tensor


def unwrap_model(model):
    return model.module if hasattr(model, "module") else model


def resolve_precision(requested_precision: str, device_kind: str) -> str:
    if requested_precision == "bf16-mixed":
        if device_kind == "cuda" and torch.cuda.is_available() and torch.cuda.is_bf16_supported():
            return requested_precision
        if is_main_process():
            print(f'Falling back from precision={requested_precision!r} to "32-true" because bf16 is not available in this runtime.')
        return "32-true"
    if requested_precision == "16-mixed" and device_kind != "cuda":
        if is_main_process():
            print(f'Falling back from precision={requested_precision!r} to "32-true" because fp16 autocast needs CUDA.')
        return "32-true"
    return requested_precision


def setup_distributed() -> torch.device:
    world_size = get_world_size()
    if USE_DDP and world_size > 1:
        if not torch.cuda.is_available():
            raise RuntimeError("DDP was requested, but CUDA is not available.")
        if not is_dist_available_and_initialized():
            dist.init_process_group(backend="nccl")
        local_rank = get_local_rank()
        torch.cuda.set_device(local_rank)
        return torch.device("cuda", local_rank)
    return torch.device("cuda", 0) if torch.cuda.is_available() else torch.device("cpu")


def cleanup_distributed() -> None:
    if is_dist_available_and_initialized():
        dist.destroy_process_group()


def resolve_path_against_roots(value: str, roots: List[Path]) -> Path:
    raw_path = Path(str(value)).expanduser()
    if raw_path.is_absolute():
        return raw_path.resolve()
    return (roots[0] / raw_path).resolve()


def resolve_run_manifest_artifact_path(
    run_manifest: dict,
    key: str,
    run_data_dir: Path,
    base_artifact_dir: Path,
    aliases: Optional[List[str]] = None,
) -> Path:
    artifacts = get_manifest_mapping(run_manifest, "artifacts")
    value, _ = get_manifest_mapping_value(artifacts, manifest_value_candidates(key, aliases))
    if value is None:
        raise KeyError(
            f"Run-data manifest is missing required artifact key {key!r}. "
            f"Available artifact keys: {sorted(artifacts)}"
        )
    return resolve_path_against_roots(str(value), [run_data_dir, base_artifact_dir])


def validate_run_data_manifest(run_manifest: dict) -> None:
    schema_version = int(run_manifest.get("schema_version", 0))
    if schema_version < 1:
        raise ValueError(
            "run_data_manifest.json has an unsupported schema_version. "
            "Re-run 01_prepare_run_data_cache.ipynb."
        )
    required_top_level = [
        "base_prep_manifest_path",
        "base_artifact_dir",
        "train_triples_sample",
        "seed",
        "dev_eval_mode",
        "dev_eval_seed",
        "counts",
        "artifacts",
    ]
    missing_top_level = [key for key in required_top_level if run_manifest.get(key) is None]
    if missing_top_level:
        raise ValueError(
            "run_data_manifest.json is missing required top-level keys. "
            f"Missing: {missing_top_level}"
        )
    required_artifact_keys = [
        "sampled_train_triples_tsv",
        "train_query_tokens_pkl",
        "train_passage_tokens_pkl",
        "dev_query_tokens_pkl",
        "epoch_dev_candidates_pkl",
        "epoch_dev_qrels_pkl",
        "passage_token_shards_dir",
        "passage_token_shards_index_json",
        "passage_token_store_stats_json",
    ]
    if bool(run_manifest.get("run_final_full_dev", True)):
        required_artifact_keys.extend(["final_dev_candidates_pkl", "final_dev_qrels_pkl"])
    artifacts = get_manifest_mapping(run_manifest, "artifacts")
    missing_artifacts = [key for key in required_artifact_keys if artifacts.get(key) is None]
    if missing_artifacts:
        raise ValueError(
            "run_data_manifest.json is missing required artifact entries. "
            f"Missing: {missing_artifacts}. Available: {sorted(artifacts)}"
        )


def validate_run_data_compatibility(run_manifest: dict) -> None:
    mismatches = {}
    expected = {
        "tokenizer_name": TOKENIZER_NAME,
        "seq_len": int(SEQ_LEN),
        "max_query_len": int(MAX_QUERY_LEN),
        "max_doc_len": int(MAX_DOC_LEN),
        "train_triples_sample": int(TRAIN_TRIPLES_SAMPLE),
        "seed": int(SEED),
        "dev_eval_mode": str(DEV_EVAL_MODE),
        "dev_eval_seed": int(DEV_EVAL_SEED),
    }
    actual = {
        "tokenizer_name": str(run_manifest.get("tokenizer_name", "")),
        "seq_len": int(run_manifest.get("seq_len", -1)),
        "max_query_len": int(run_manifest.get("max_query_len", -1)),
        "max_doc_len": int(run_manifest.get("max_doc_len", -1)),
        "train_triples_sample": int(run_manifest.get("train_triples_sample", -1)),
        "seed": int(run_manifest.get("seed", -1)),
        "dev_eval_mode": str(run_manifest.get("dev_eval_mode", "")),
        "dev_eval_seed": int(run_manifest.get("dev_eval_seed", -1)),
    }
    for key, expected_value in expected.items():
        if actual[key] != expected_value:
            mismatches[key] = {"expected": expected_value, "actual": actual[key]}
    if DEV_EVAL_MODE == "quick_fraction":
        expected_fraction = float(DEV_EVAL_FRACTION)
        actual_fraction = float(run_manifest.get("dev_eval_fraction", -1.0))
        if abs(expected_fraction - actual_fraction) > 1e-12:
            mismatches["dev_eval_fraction"] = {"expected": expected_fraction, "actual": actual_fraction}
    if mismatches:
        raise ValueError(
            "run_data_manifest.json is incompatible with the current training config. "
            f"Mismatches: {mismatches}. Re-run 01_prepare_run_data_cache.ipynb with matching settings."
        )


def load_passage_token_shard_index(index_path: Path) -> dict:
    return json.loads(index_path.read_text())


def build_passage_token_subset_loader(
    index_path: Path,
    artifact_dir: Path,
    shards_dir_path: Optional[Path] = None,
    shard_cache_size: int = 8,
):
    index = load_passage_token_shard_index(index_path)
    if index.get("format") != "sharded_flat_token_arrays_v1":
        raise ValueError(
            f"Unsupported passage token store format: {index.get('format')!r}. "
            "This notebook expects a sharded flat token store described by passage_token_shards_index_json."
        )

    shard_entries = index.get("shards") or []
    if not shard_entries:
        raise ValueError(f"Passage token shard index is missing shard entries: {index_path}")

    shard_size = int(index["shard_size"])
    shards_by_id = {int(entry["shard_id"]): entry for entry in shard_entries}
    resolved_shards_dir = Path(shards_dir_path).resolve() if shards_dir_path is not None else None

    def resolve_shard_path(shard_entry: dict) -> Path:
        candidate_paths: List[Path] = []
        raw_path_value = shard_entry.get("path")
        if raw_path_value:
            raw_path = Path(str(raw_path_value)).expanduser()
            if raw_path.is_absolute():
                candidate_paths.append(raw_path)
                candidate_paths.append(artifact_dir / raw_path.name)
                if resolved_shards_dir is not None:
                    candidate_paths.append(resolved_shards_dir / raw_path.name)
            else:
                if resolved_shards_dir is not None:
                    candidate_paths.append(resolved_shards_dir / raw_path)
                candidate_paths.append(artifact_dir / raw_path)

        for key in ("filename", "file_name", "basename", "name"):
            filename = shard_entry.get(key)
            if filename and resolved_shards_dir is not None:
                candidate_paths.append(resolved_shards_dir / str(filename))

        seen_candidates = set()
        unique_candidates: List[Path] = []
        for candidate in candidate_paths:
            candidate = candidate.expanduser()
            candidate_key = str(candidate)
            if candidate_key in seen_candidates:
                continue
            seen_candidates.add(candidate_key)
            unique_candidates.append(candidate)
            if candidate.is_file():
                return candidate.resolve()

        raise FileNotFoundError(
            "Passage token shard file is missing. "
            f"Checked shard_id={shard_entry.get('shard_id')} candidates: {[str(path) for path in unique_candidates]}"
        )

    @lru_cache(maxsize=shard_cache_size)
    def load_shard(shard_id: int):
        shard_entry = shards_by_id.get(int(shard_id))
        if shard_entry is None:
            raise KeyError(f"No passage token shard found for shard_id={shard_id}")
        shard_path = resolve_shard_path(shard_entry)
        with np.load(shard_path, allow_pickle=False) as shard_data:
            shard_pids = shard_data["pid"]
            shard_offsets = shard_data["offsets"]
            shard_token_ids = shard_data["token_ids"]
        pid_lookup = {int(pid): idx for idx, pid in enumerate(shard_pids.tolist())}
        return {
            "pid": shard_pids,
            "offsets": shard_offsets,
            "token_ids": shard_token_ids,
            "pid_lookup": pid_lookup,
        }

    def get_passage_tokens(pid: int) -> List[int]:
        shard = load_shard(int(pid) // shard_size)
        pid_idx = shard["pid_lookup"].get(int(pid))
        if pid_idx is None:
            raise KeyError(f"Missing cached passage tokens for pid={pid}")
        start = int(shard["offsets"][pid_idx])
        end = int(shard["offsets"][pid_idx + 1])
        return shard["token_ids"][start:end].tolist()

    def load_passage_tokens_for_subset(pid_list: Iterable[int]) -> Dict[int, List[int]]:
        unique_pids = sorted({int(pid) for pid in pid_list})
        pids_by_shard: Dict[int, List[int]] = {}
        for pid in unique_pids:
            pids_by_shard.setdefault(int(pid) // shard_size, []).append(int(pid))

        subset: Dict[int, List[int]] = {}
        with tqdm(
            total=len(unique_pids),
            desc="Load cached train passages",
            disable=not is_main_process(),
        ) as pbar:
            for shard_id in sorted(pids_by_shard):
                shard = load_shard(shard_id)
                for pid in pids_by_shard[shard_id]:
                    pid_idx = shard["pid_lookup"].get(pid)
                    if pid_idx is None:
                        raise KeyError(f"Missing cached passage tokens for pid={pid}")
                    start = int(shard["offsets"][pid_idx])
                    end = int(shard["offsets"][pid_idx + 1])
                    subset[pid] = shard["token_ids"][start:end].tolist()
                    pbar.update(1)
        return subset

    return index, get_passage_tokens, load_passage_tokens_for_subset


seed_everything(SEED)

AVAILABLE_CUDA_DEVICES = torch.cuda.device_count() if torch.cuda.is_available() else 0
WORLD_SIZE_ENV = int(os.environ.get("WORLD_SIZE", "1"))
DEVICE_KIND = "cuda" if torch.cuda.is_available() else "cpu"
SHOULD_USE_DDP = USE_DDP and WORLD_SIZE_ENV > 1
if USE_DDP and WORLD_SIZE_ENV == 1 and is_main_process():
    print("USE_DDP=True but WORLD_SIZE=1. Run with torchrun for real multi-GPU training.")
    print("Falling back to single-process training.")
if DEVICES > AVAILABLE_CUDA_DEVICES and AVAILABLE_CUDA_DEVICES > 0 and is_main_process():
    print(f"Requested DEVICES={DEVICES}, but only {AVAILABLE_CUDA_DEVICES} CUDA devices are visible in this runtime.")

EFFECTIVE_PRECISION = resolve_precision(PRECISION, DEVICE_KIND)
MODEL_FORWARD_DTYPE = "bfloat16" if EFFECTIVE_PRECISION == "bf16-mixed" else "float32"
ACCELERATOR = "gpu" if DEVICE_KIND == "cuda" else "cpu"
TRAINER_DEVICES = WORLD_SIZE_ENV if SHOULD_USE_DDP else (min(max(1, DEVICES), AVAILABLE_CUDA_DEVICES) if ACCELERATOR == "gpu" else 1)
STRATEGY = "ddp" if SHOULD_USE_DDP else "single_process"

RUN_DIR = OUTPUT_ROOT / "runs" / EXPERIMENT_NAME
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
EVAL_DIR = RUN_DIR / "eval"
EPOCH_EVAL_DIR = EVAL_DIR / "epochs"
FINAL_EVAL_DIR = EVAL_DIR / "final"
TRAIN_LOG_PATH = LOG_DIR / "train_metrics.csv"
TRAINING_CONFIG_PATH = RUN_DIR / "training_config.json"
FIT_SUMMARY_PATH = RUN_DIR / "fit_summary.json"
RUN_ARTIFACTS_PATH = RUN_DIR / "run_artifacts.json"
EPOCH_SUMMARIES_PATH = LOG_DIR / "epoch_summaries.json"
DEV_METRICS_BY_EPOCH_PATH = LOG_DIR / "dev_metrics_by_epoch.csv"
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_checkpoint.pt"
BEST_TRAIN_LOSS_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_train_loss.pt"
BEST_DEV_MRR10_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_dev_mrr10.pt"
FINAL_RUN_PATH = FINAL_EVAL_DIR / "final_dev_best.run"
FINAL_METRICS_PATH = FINAL_EVAL_DIR / "final_dev_metrics.json"
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EPOCH_EVAL_DIR.mkdir(parents=True, exist_ok=True)
FINAL_EVAL_DIR.mkdir(parents=True, exist_ok=True)

RUN_DATA_MANIFEST_PATH = find_run_data_manifest(INPUT_RUN_DATA_MANIFEST_PATH, OUTPUT_ROOT)
run_data_manifest = json.loads(RUN_DATA_MANIFEST_PATH.read_text())
RUN_DATA_CACHE_DIR = RUN_DATA_MANIFEST_PATH.parent.resolve()
base_artifact_dir_value = run_data_manifest.get("base_artifact_dir")
if not base_artifact_dir_value:
    raise ValueError("run_data_manifest.json is missing base_artifact_dir")
ARTIFACT_DIR = resolve_path_against_roots(str(base_artifact_dir_value), [RUN_DATA_CACHE_DIR])
base_prep_manifest_value = run_data_manifest.get("base_prep_manifest_path") or MANUAL_PREP_MANIFEST_PATH
if not base_prep_manifest_value:
    raise ValueError(
        "run_data_manifest.json does not provide base_prep_manifest_path and INPUT_PREP_MANIFEST_PATH is empty."
    )
PREP_MANIFEST_PATH = resolve_path_against_roots(str(base_prep_manifest_value), [RUN_DATA_CACHE_DIR, ARTIFACT_DIR])
manifest = json.loads(PREP_MANIFEST_PATH.read_text())

tokenizer_local_value = manifest.get("tokenizer_local_path", "tokenizer")
tokenizer_local_path = resolve_relative_or_absolute_path(tokenizer_local_value, ARTIFACT_DIR)
sampled_train_triples_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "sampled_train_triples_tsv",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
train_query_tokens_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "train_query_tokens_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
train_passage_tokens_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "train_passage_tokens_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
dev_query_tokens_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "dev_query_tokens_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
epoch_dev_candidates_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "epoch_dev_candidates_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
epoch_dev_qrels_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "epoch_dev_qrels_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
final_dev_candidates_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "final_dev_candidates_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
final_dev_qrels_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "final_dev_qrels_pkl",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
)
passage_token_shards_dir_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "passage_token_shards_dir",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
    aliases=["passage_tokens_shards_dir"],
)
passage_token_shards_index_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "passage_token_shards_index_json",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
    aliases=["passage_tokens_store_index_json"],
)
passage_token_store_stats_path = resolve_run_manifest_artifact_path(
    run_data_manifest,
    "passage_token_store_stats_json",
    RUN_DATA_CACHE_DIR,
    ARTIFACT_DIR,
    aliases=["passage_tokens_store_stats_json"],
)


tokenizer_name = manifest.get("tokenizer_name", TOKENIZER_NAME)
if tokenizer_local_path.exists():
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_local_path, use_fast=True, local_files_only=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast=True)

train_query_tokens = load_pickle(train_query_tokens_path)
train_passage_tokens = load_pickle(train_passage_tokens_path)
dev_query_tokens = load_pickle(dev_query_tokens_path)
epoch_dev_candidates_artifact = load_pickle(epoch_dev_candidates_path)
epoch_dev_qrels_artifact = load_pickle(epoch_dev_qrels_path)
final_dev_candidates = load_pickle(final_dev_candidates_path) if RUN_FINAL_FULL_DEV else None
final_dev_qrels = load_pickle(final_dev_qrels_path) if RUN_FINAL_FULL_DEV else None

passage_token_store_index, get_cached_passage_tokens, _load_passage_tokens_for_subset = build_passage_token_subset_loader(
    passage_token_shards_index_path,
    ARTIFACT_DIR,
    shards_dir_path=passage_token_shards_dir_path,
)

RUN_DATA_COUNTS = get_manifest_mapping(run_data_manifest, "counts")
sampled_train_triples_count = int(RUN_DATA_COUNTS.get("sampled_train_triples", 0))
sampled_train_queries_count = int(RUN_DATA_COUNTS.get("sampled_train_queries", len(train_query_tokens)))
sampled_train_passages_count = int(RUN_DATA_COUNTS.get("sampled_train_passages", len(train_passage_tokens)))
epoch_dev_queries_count = int(RUN_DATA_COUNTS.get("epoch_dev_queries", len(epoch_dev_candidates_artifact["qid_order"])))
epoch_dev_candidate_rows_count = int(RUN_DATA_COUNTS.get("epoch_dev_candidate_rows", len(epoch_dev_candidates_artifact["pid"])))
RUN_DATA_EPOCH_DEV_MODE_LABEL = str(run_data_manifest.get("epoch_dev_mode_label") or run_data_manifest.get("dev_eval_mode"))

SEQ_LEN = int(manifest["seq_len"])
MAX_QUERY_LEN = int(manifest["max_query_len"])
MAX_DOC_LEN = int(manifest["max_doc_len"])

CLS_ID = tokenizer.cls_token_id
SEP_ID = tokenizer.sep_token_id
PAD_ID = tokenizer.pad_token_id
if CLS_ID is None or SEP_ID is None or PAD_ID is None:
    raise ValueError("Tokenizer must provide cls_token_id, sep_token_id, and pad_token_id")

model_config = {
    "batch_size": PER_DEVICE_BATCH_SIZE,
    "seq_len": SEQ_LEN,
    "vocab_size": len(tokenizer),
    "num_layers": VANILLA_TRANSFORMER_LAYERS,
    "hidden_size": HIDDEN_SIZE,
    "expansion": EXPANSION,
    "num_heads": NUM_HEADS,
    "pos_encodings": POS_ENCODINGS,
    "halt_max_steps": HALT_MAX_STEPS,
    "forward_dtype": MODEL_FORWARD_DTYPE,
    "num_segment_types": 3,
}

prep_summary = {
    "run_profile": RUN_PROFILE,
    "expected_cache_tag": EXPECTED_CACHE_TAG,
    "manifest_cache_tag": manifest.get("cache_tag"),
    "prep_manifest_path": str(PREP_MANIFEST_PATH),
    "run_data_manifest_path": str(RUN_DATA_MANIFEST_PATH),
    "artifact_dir": str(ARTIFACT_DIR),
    "run_data_cache_dir": str(RUN_DATA_CACHE_DIR),
    "output_root": str(OUTPUT_ROOT),
    "accelerator": ACCELERATOR,
    "requested_devices": DEVICES,
    "trainer_devices": TRAINER_DEVICES,
    "available_cuda_devices": AVAILABLE_CUDA_DEVICES,
    "strategy": STRATEGY,
    "precision": EFFECTIVE_PRECISION,
    "run_dir": str(RUN_DIR),
    "dev_eval_mode": "final_full_only",
    "dev_eval_fraction": None,
    "dev_eval_seed": None,
    "epoch_dev_eval_enabled": False,
    "final_dev_eval_only": True,
    "run_final_full_dev": RUN_FINAL_FULL_DEV,
    "sampled_train_triples_path": str(sampled_train_triples_path),
    "sampled_train_triples": sampled_train_triples_count,
    "sampled_train_queries": sampled_train_queries_count,
    "sampled_train_passages": sampled_train_passages_count,
    "epoch_dev_queries": epoch_dev_queries_count,
    "epoch_dev_candidate_rows": epoch_dev_candidate_rows_count,
    "epoch_dev_mode_label": RUN_DATA_EPOCH_DEV_MODE_LABEL,
    "passage_token_store_format": passage_token_store_index.get("format"),
    "passage_token_shard_count": int(passage_token_store_index.get("shard_count", len(passage_token_store_index.get("shards", [])))),
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "freeze_token_embeddings": FREEZE_TOKEN_EMBEDDINGS,
    "freeze_segment_embeddings": FREEZE_SEGMENT_EMBEDDINGS,
}
run_artifacts = {
    "prep_manifest_path": str(PREP_MANIFEST_PATH),
    "run_data_manifest_path": str(RUN_DATA_MANIFEST_PATH),
    "artifact_dir": str(ARTIFACT_DIR),
    "run_data_cache_dir": str(RUN_DATA_CACHE_DIR),
    "output_root": str(OUTPUT_ROOT),
    "run_dir": str(RUN_DIR),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "log_dir": str(LOG_DIR),
    "eval_dir": str(EVAL_DIR),
    "epoch_eval_dir": str(EPOCH_EVAL_DIR),
    "final_eval_dir": str(FINAL_EVAL_DIR),
    "dev_eval_mode": "final_full_only",
    "epoch_dev_mode_label": "final_only",
    "dev_eval_fraction": None,
    "dev_eval_seed": None,
    "epoch_dev_eval_enabled": False,
    "final_dev_eval_only": True,
    "train_triples_sample": TRAIN_TRIPLES_SAMPLE,
    "sampled_train_triples_path": str(sampled_train_triples_path),
    "passage_token_shards_index_path": str(passage_token_shards_index_path),
    "passage_token_store_stats_path": str(passage_token_store_stats_path),
    "counts": {
        "sampled_train_triples": sampled_train_triples_count,
        "sampled_train_queries": sampled_train_queries_count,
        "sampled_train_passages": sampled_train_passages_count,
        "epoch_dev_queries": epoch_dev_queries_count,
        "epoch_dev_candidate_rows": epoch_dev_candidate_rows_count,
    },
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "freeze_token_embeddings": FREEZE_TOKEN_EMBEDDINGS,
    "freeze_segment_embeddings": FREEZE_SEGMENT_EMBEDDINGS,
    "best_dev_mrr10_checkpoint_path": None,
    "best_train_loss_checkpoint_path": str(BEST_TRAIN_LOSS_CHECKPOINT_PATH),
    "last_checkpoint_path": str(LAST_CHECKPOINT_PATH),
    "final_metrics_path": str(FINAL_METRICS_PATH),
}
print(json.dumps(prep_summary, indent=2))
prep_summary


In [ ]:
def encode_pair(query_tokens: List[int], passage_tokens: List[int], seq_len: int, max_query_len: int, max_doc_len: int) -> Dict[str, List[int]]:
    q_tokens = list(query_tokens[:max_query_len])
    d_tokens = list(passage_tokens[:max_doc_len])

    input_ids = [CLS_ID] + q_tokens + [SEP_ID] + d_tokens + [SEP_ID]
    token_type_ids = [0] + [1] * len(q_tokens) + [0] + [2] * len(d_tokens) + [0]
    attention_mask = [1] * len(input_ids)

    pad_len = seq_len - len(input_ids)
    if pad_len < 0:
        raise ValueError(f'Pair length {len(input_ids)} exceeds configured seq_len={seq_len}')

    input_ids += [PAD_ID] * pad_len
    token_type_ids += [0] * pad_len
    attention_mask += [0] * pad_len

    return {
        'input_ids': input_ids,
        'token_type_ids': token_type_ids,
        'attention_mask': attention_mask,
    }


def collate_encoded_pairs(encoded_pairs: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
    batch = {
        key: torch.tensor([item[key] for item in encoded_pairs], dtype=torch.long)
        for key in ['input_ids', 'token_type_ids', 'attention_mask']
    }
    return batch


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device) for key, value in batch.items()}


def reciprocal_rank_at_k(ranked_pids: List[int], relevant_pids: set, k: int = 10) -> float:
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        if pid in relevant_pids:
            return 1.0 / rank
    return 0.0


def iter_grouped_candidates(candidates_artifact, query_limit: int = None):
    qid_order = candidates_artifact['qid_order']
    qid_offsets = candidates_artifact['qid_offsets']
    pid_values = candidates_artifact['pid']
    bm25_ranks = candidates_artifact['bm25_rank']
    limit = len(qid_order) if query_limit is None else min(len(qid_order), query_limit)
    for idx in range(limit):
        qid = int(qid_order[idx])
        start = int(qid_offsets[idx])
        end = int(qid_offsets[idx + 1])
        yield qid, pid_values[start:end], bm25_ranks[start:end]


In [ ]:
CosSin = Tuple[torch.Tensor, torch.Tensor]


def trunc_normal_init_(tensor: torch.Tensor, std: float = 1.0, lower: float = -2.0, upper: float = 2.0):
    with torch.no_grad():
        if std == 0:
            tensor.zero_()
        else:
            sqrt2 = math.sqrt(2)
            a = math.erf(lower / sqrt2)
            b = math.erf(upper / sqrt2)
            z = (b - a) / 2

            c = (2 * math.pi) ** -0.5
            pdf_u = c * math.exp(-0.5 * lower ** 2)
            pdf_l = c * math.exp(-0.5 * upper ** 2)
            comp_std = std / math.sqrt(1 - (upper * pdf_u - lower * pdf_l) / z - ((pdf_u - pdf_l) / z) ** 2)

            tensor.uniform_(a, b)
            tensor.erfinv_()
            tensor.mul_(sqrt2 * comp_std)
            tensor.clip_(lower * comp_std, upper * comp_std)
    return tensor


def _find_multiple(a: int, b: int) -> int:
    return (-(a // -b)) * b


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    seq_len = q.shape[1]
    cos = cos[:seq_len]
    sin = sin[:seq_len]
    orig_dtype = q.dtype
    q = q.to(cos.dtype)
    k = k.to(cos.dtype)
    q_embed = (q * cos.unsqueeze(-2)) + (rotate_half(q) * sin.unsqueeze(-2))
    k_embed = (k * cos.unsqueeze(-2)) + (rotate_half(k) * sin.unsqueeze(-2))
    return q_embed.to(orig_dtype), k_embed.to(orig_dtype)


class CastedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool):
        super().__init__()
        self.weight = nn.Parameter(trunc_normal_init_(torch.empty((out_features, in_features)), std=1.0 / (in_features ** 0.5)))
        self.bias = nn.Parameter(torch.zeros((out_features,))) if bias else None

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        bias = self.bias.to(inputs.dtype) if self.bias is not None else None
        return F.linear(inputs, self.weight.to(inputs.dtype), bias=bias)


class CastedEmbedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int, init_std: float, cast_to: torch.dtype):
        super().__init__()
        self.cast_to = cast_to
        self.embedding_weight = nn.Parameter(trunc_normal_init_(torch.empty((num_embeddings, embedding_dim)), std=init_std))

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return F.embedding(inputs, self.embedding_weight.to(self.cast_to))


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int, base: float):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        t = torch.arange(max_position_embeddings, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer('cos_cached', emb.cos(), persistent=False)
        self.register_buffer('sin_cached', emb.sin(), persistent=False)

    def forward(self) -> CosSin:
        return self.cos_cached, self.sin_cached


class Attention(nn.Module):
    def __init__(self, hidden_size: int, head_dim: int, num_heads: int, num_key_value_heads: int, causal: bool = False):
        super().__init__()
        self.hidden_size = hidden_size
        self.head_dim = head_dim
        self.output_size = head_dim * num_heads
        self.num_heads = num_heads
        self.num_key_value_heads = num_key_value_heads
        self.causal = causal
        self.qkv_proj = CastedLinear(hidden_size, (num_heads + 2 * num_key_value_heads) * head_dim, bias=False)
        self.o_proj = CastedLinear(self.output_size, hidden_size, bias=False)

    def forward(self, cos_sin: CosSin, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        batch_size, seq_len, _ = hidden_states.shape
        qkv = self.qkv_proj(hidden_states)
        qkv = qkv.view(batch_size, seq_len, self.num_heads + 2 * self.num_key_value_heads, self.head_dim)
        query = qkv[:, :, : self.num_heads]
        key = qkv[:, :, self.num_heads : self.num_heads + self.num_key_value_heads]
        value = qkv[:, :, self.num_heads + self.num_key_value_heads :]
        if cos_sin is not None:
            cos, sin = cos_sin
            query, key = apply_rotary_pos_emb(query, key, cos, sin)
        query = query.permute(0, 2, 1, 3)
        key = key.permute(0, 2, 1, 3)
        value = value.permute(0, 2, 1, 3)
        attn_mask = attention_mask.to(query.dtype) if attention_mask is not None else None
        attn_output = scaled_dot_product_attention(query=query, key=key, value=value, attn_mask=attn_mask, is_causal=self.causal)
        attn_output = attn_output.permute(0, 2, 1, 3).reshape(batch_size, seq_len, self.output_size)
        return self.o_proj(attn_output)


class SwiGLU(nn.Module):
    def __init__(self, hidden_size: int, expansion: float):
        super().__init__()
        inter = _find_multiple(round(expansion * hidden_size * 2 / 3), 256)
        self.gate_up_proj = CastedLinear(hidden_size, inter * 2, bias=False)
        self.down_proj = CastedLinear(inter, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate, up = self.gate_up_proj(x).chunk(2, dim=-1)
        return self.down_proj(F.silu(gate) * up)


def rms_norm(hidden_states: torch.Tensor, variance_epsilon: float) -> torch.Tensor:
    input_dtype = hidden_states.dtype
    hidden_states = hidden_states.to(torch.float32)
    variance = hidden_states.square().mean(-1, keepdim=True)
    hidden_states = hidden_states * torch.rsqrt(variance + variance_epsilon)
    return hidden_states.to(input_dtype)


In [ ]:
@dataclass
class VanillaTransformerRerankerCarry:
    halted: torch.Tensor


@dataclass
class VanillaTransformerRerankerConfig:
    batch_size: int
    seq_len: int
    vocab_size: int
    num_layers: int
    hidden_size: int
    expansion: float
    num_heads: int
    pos_encodings: str
    rms_norm_eps: float = 1e-5
    rope_theta: float = 10000.0
    halt_max_steps: int = 1
    forward_dtype: str = 'float32'
    num_segment_types: int = 3


class VanillaTransformerBlock(nn.Module):
    def __init__(self, config: VanillaTransformerRerankerConfig) -> None:
        super().__init__()
        self.self_attn = Attention(
            hidden_size=config.hidden_size,
            head_dim=config.hidden_size // config.num_heads,
            num_heads=config.num_heads,
            num_key_value_heads=config.num_heads,
            causal=False,
        )
        self.mlp = SwiGLU(hidden_size=config.hidden_size, expansion=config.expansion)
        self.norm_eps = config.rms_norm_eps

    def forward(self, hidden_states: torch.Tensor, cos_sin: CosSin = None, attention_mask: torch.Tensor = None) -> torch.Tensor:
        hidden_states = rms_norm(
            hidden_states + self.self_attn(cos_sin=cos_sin, hidden_states=hidden_states, attention_mask=attention_mask),
            variance_epsilon=self.norm_eps,
        )
        hidden_states = rms_norm(hidden_states + self.mlp(hidden_states), variance_epsilon=self.norm_eps)
        return hidden_states


class VanillaTransformerRerankerInner(nn.Module):
    def __init__(self, config: VanillaTransformerRerankerConfig) -> None:
        super().__init__()
        self.config = config
        self.forward_dtype = getattr(torch, self.config.forward_dtype)
        self.embed_scale = math.sqrt(self.config.hidden_size)
        embed_init_std = 1.0 / self.embed_scale

        self.embed_tokens = CastedEmbedding(self.config.vocab_size, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.segment_emb = CastedEmbedding(self.config.num_segment_types, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)
        self.score_head = CastedLinear(self.config.hidden_size, 1, bias=True)
        self.layers = nn.ModuleList([VanillaTransformerBlock(self.config) for _ in range(self.config.num_layers)])

        if self.config.pos_encodings == 'rope':
            self.rotary_emb = RotaryEmbedding(
                dim=self.config.hidden_size // self.config.num_heads,
                max_position_embeddings=self.config.seq_len,
                base=self.config.rope_theta,
            )
        elif self.config.pos_encodings == 'learned':
            self.embed_pos = CastedEmbedding(self.config.seq_len, self.config.hidden_size, init_std=embed_init_std, cast_to=self.forward_dtype)

    def _input_embeddings(self, input_ids: torch.Tensor, token_type_ids: torch.Tensor) -> torch.Tensor:
        embedding = self.embed_tokens(input_ids.to(torch.int64))
        embedding = embedding + self.segment_emb(token_type_ids.to(torch.int64))
        if self.config.pos_encodings == 'learned':
            embedding = 0.707106781 * (embedding + self.embed_pos.embedding_weight[: input_ids.shape[1]].to(self.forward_dtype))
        return self.embed_scale * embedding

    def _attention_mask(self, attention_mask: torch.Tensor) -> torch.Tensor:
        attention_mask = attention_mask.to(torch.bool)
        additive_mask = torch.zeros(attention_mask.shape, dtype=self.forward_dtype, device=attention_mask.device)
        additive_mask = additive_mask.masked_fill(~attention_mask, torch.finfo(self.forward_dtype).min)
        return additive_mask[:, None, None, :]

    def forward(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        cos_sin = self.rotary_emb() if hasattr(self, 'rotary_emb') else None
        attention_mask = self._attention_mask(batch['attention_mask'])
        hidden_states = self._input_embeddings(batch['input_ids'], batch['token_type_ids'])
        for layer in self.layers:
            hidden_states = layer(hidden_states=hidden_states, cos_sin=cos_sin, attention_mask=attention_mask)
        cls_state = hidden_states[:, 0]
        scores = self.score_head(cls_state).squeeze(-1)
        return {'scores': scores}


class VanillaTransformerReranker(nn.Module):
    def __init__(self, config_dict: dict):
        super().__init__()
        self.config = VanillaTransformerRerankerConfig(**config_dict)
        self.inner = VanillaTransformerRerankerInner(self.config)

    def initial_carry(self, batch: Dict[str, torch.Tensor]) -> VanillaTransformerRerankerCarry:
        batch_size = batch['input_ids'].shape[0]
        return VanillaTransformerRerankerCarry(
            halted=torch.ones((batch_size,), dtype=torch.bool, device=batch['input_ids'].device),
        )

    def forward(self, carry: VanillaTransformerRerankerCarry, batch: Dict[str, torch.Tensor]):
        del carry
        outputs = self.inner(batch)
        batch_size = batch['input_ids'].shape[0]
        new_carry = VanillaTransformerRerankerCarry(
            halted=torch.ones((batch_size,), dtype=torch.bool, device=batch['input_ids'].device),
        )
        return new_carry, outputs


In [ ]:
class PairwiseTripleDataset(Dataset):
    def __init__(self, triples_path: Path, query_token_map: Dict[int, List[int]], passage_token_map: Dict[int, List[int]]):
        self.query_token_map = query_token_map
        self.passage_token_map = passage_token_map
        self.triples: List[Tuple[int, int, int]] = []
        with triples_path.open("r", encoding="utf-8", newline="") as handle:
            reader = csv.reader(handle, delimiter="\t")
            for row in tqdm(reader, desc=f"Load {triples_path.name}"):
                if len(row) >= 3:
                    self.triples.append((int(row[0]), int(row[1]), int(row[2])))

    def __len__(self) -> int:
        return len(self.triples)

    def __getitem__(self, index: int):
        qid, pos_pid, neg_pid = self.triples[index]
        if qid not in self.query_token_map:
            raise KeyError(f"Missing query tokens for qid={qid}")
        if pos_pid not in self.passage_token_map or neg_pid not in self.passage_token_map:
            raise KeyError(f"Missing cached passage tokens for one of {pos_pid}, {neg_pid}")
        return {
            "qid": qid,
            "query_tokens": self.query_token_map[qid],
            "pos_tokens": self.passage_token_map[pos_pid],
            "neg_tokens": self.passage_token_map[neg_pid],
        }


def pairwise_collate(batch_items: List[Dict[str, object]]):
    pos_pairs = [encode_pair(item["query_tokens"], item["pos_tokens"], SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN) for item in batch_items]
    neg_pairs = [encode_pair(item["query_tokens"], item["neg_tokens"], SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN) for item in batch_items]
    return collate_encoded_pairs(pos_pairs), collate_encoded_pairs(neg_pairs)


def run_model_once(model, batch: Dict[str, torch.Tensor]):
    stateful_model = unwrap_model(model)
    carry = stateful_model.initial_carry(batch)
    outputs = None
    for _ in range(stateful_model.config.halt_max_steps):
        carry, outputs = model(carry, batch)
        if bool(carry.halted.all()):
            break
    if outputs is None:
        raise RuntimeError("Model produced no outputs")
    return outputs["scores"], outputs


def build_warmup_steps(total_steps: int, warmup_ratio: float) -> int:
    if total_steps <= 0:
        return 0
    return max(0, min(total_steps - 1, int(math.ceil(total_steps * warmup_ratio))))


def build_linear_warmup_decay_lambda(warmup_steps: int, total_steps: Optional[int]):
    def lr_lambda(current_step: int) -> float:
        if total_steps is None or total_steps <= 0:
            return 1.0
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 1.0 - progress)

    return lr_lambda


def compute_pairwise_batch_metrics(model, pos_batch, neg_batch):
    pos_scores, _ = run_model_once(model, pos_batch)
    neg_scores, _ = run_model_once(model, neg_batch)
    margin = pos_scores - neg_scores
    loss = -F.logsigmoid(margin).mean()
    pairwise_acc = (pos_scores > neg_scores).float().mean()
    return {
        "loss": loss,
        "pairwise_acc": pairwise_acc,
        "margin": margin.mean(),
        "pos_score": pos_scores.mean(),
        "neg_score": neg_scores.mean(),
    }


train_dataset = PairwiseTripleDataset(sampled_train_triples_path, train_query_tokens, train_passage_tokens)
WORLD_SIZE = WORLD_SIZE_ENV if SHOULD_USE_DDP else 1
train_sampler = None
train_shuffle = True
if SHOULD_USE_DDP:
    train_sampler = DistributedSampler(
        train_dataset,
        num_replicas=WORLD_SIZE,
        rank=get_rank(),
        shuffle=True,
        drop_last=False,
    )
    train_shuffle = False

train_loader = DataLoader(
    train_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    sampler=train_sampler,
    shuffle=train_shuffle,
    num_workers=NUM_WORKERS,
    collate_fn=pairwise_collate,
    pin_memory=torch.cuda.is_available(),
)

GLOBAL_BATCH_SIZE = PER_DEVICE_BATCH_SIZE * WORLD_SIZE * GRAD_ACCUM_STEPS
ESTIMATED_STEPS_PER_EPOCH = max(1, math.ceil(len(train_loader) / GRAD_ACCUM_STEPS))
ESTIMATED_TOTAL_STEPS = ESTIMATED_STEPS_PER_EPOCH * EPOCHS
if MAX_TRAIN_STEPS is not None:
    ESTIMATED_TOTAL_STEPS = min(ESTIMATED_TOTAL_STEPS, int(MAX_TRAIN_STEPS))
WARMUP_STEPS = build_warmup_steps(ESTIMATED_TOTAL_STEPS, WARMUP_RATIO)


In [ ]:
def model_device(model: nn.Module) -> torch.device:
    return next(model.parameters()).device


def get_autocast_context(device: torch.device, precision: str):
    if device.type != "cuda":
        return nullcontext()
    if precision == "bf16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    if precision == "16-mixed":
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def model_autocast_context(model: nn.Module):
    return get_autocast_context(model_device(model), EFFECTIVE_PRECISION)


def score_pid_batch(model, query_tokens: List[int], pid_batch: Iterable[int], passage_token_getter, batch_size: int):
    del batch_size
    encoded_pairs = [
        encode_pair(query_tokens, list(passage_token_getter(int(pid))), SEQ_LEN, MAX_QUERY_LEN, MAX_DOC_LEN)
        for pid in pid_batch
    ]
    batch = move_batch_to_device(collate_encoded_pairs(encoded_pairs), model_device(model))
    with torch.no_grad():
        with model_autocast_context(model):
            scores, _ = run_model_once(model, batch)
    return scores.detach().float().cpu().tolist()


RANKING_METRIC_NAMES = ["mrr@10", "hit@1", "hit@3", "hit@5", "hit@10", "ndcg@10"]


def hit_at_k(ranked_pids: List[int], relevant_pids: set, k: int) -> float:
    return 1.0 if any(pid in relevant_pids for pid in ranked_pids[:k]) else 0.0


def dcg_at_k(ranked_pids: List[int], relevant_pids: set, k: int) -> float:
    dcg = 0.0
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        rel = 1.0 if pid in relevant_pids else 0.0
        if rel:
            dcg += rel / math.log2(rank + 1)
    return dcg


def ideal_dcg_at_k(num_relevant: int, k: int) -> float:
    ideal_count = min(int(num_relevant), k)
    if ideal_count <= 0:
        return 0.0
    return sum(1.0 / math.log2(rank + 1) for rank in range(1, ideal_count + 1))


def ndcg_at_k(ranked_pids: List[int], relevant_pids: set, k: int) -> float:
    idcg = ideal_dcg_at_k(len(relevant_pids), k)
    if idcg <= 0.0:
        return 0.0
    return dcg_at_k(ranked_pids, relevant_pids, k) / idcg


def ranking_metrics_at_10(ranked_pids: List[int], relevant_pids: set) -> Dict[str, float]:
    return {
        "mrr@10": reciprocal_rank_at_k(ranked_pids, relevant_pids, k=10),
        "hit@1": hit_at_k(ranked_pids, relevant_pids, k=1),
        "hit@3": hit_at_k(ranked_pids, relevant_pids, k=3),
        "hit@5": hit_at_k(ranked_pids, relevant_pids, k=5),
        "hit@10": hit_at_k(ranked_pids, relevant_pids, k=10),
        "ndcg@10": ndcg_at_k(ranked_pids, relevant_pids, k=10),
    }


def evaluate_reranker(
    model,
    candidates_artifact,
    qrels: Dict[int, set],
    query_token_map: Dict[int, List[int]],
    run_path: Path,
    passage_token_getter,
    query_limit: int = None,
):
    was_training = model.training
    model.eval()
    run_path.parent.mkdir(parents=True, exist_ok=True)
    bm25_metric_values = {name: [] for name in RANKING_METRIC_NAMES}
    vanilla_metric_values = {name: [] for name in RANKING_METRIC_NAMES}
    evaluated_queries = 0

    total_queries = len(candidates_artifact["qid_order"])
    if query_limit is not None:
        total_queries = min(int(query_limit), total_queries)

    with run_path.open("w", encoding="utf-8") as run_handle:
        for qid, pid_values, bm25_ranks in tqdm(
            iter_grouped_candidates(candidates_artifact, query_limit=query_limit),
            total=total_queries,
            desc=f"Evaluate {run_path.name}",
        ):
            qid = int(qid)
            if qid not in query_token_map or qid not in qrels:
                continue
            candidate_pids = [int(pid) for pid in pid_values]
            candidate_bm25_ranks = [int(rank) for rank in bm25_ranks]
            if not candidate_pids:
                continue
            if len(candidate_pids) != len(candidate_bm25_ranks):
                raise ValueError(f"Candidate pid count and BM25 rank count differ for qid={qid}")
            scores: List[float] = []
            for start in range(0, len(candidate_pids), EVAL_BATCH_SIZE):
                pid_batch = candidate_pids[start:start + EVAL_BATCH_SIZE]
                scores.extend(
                    score_pid_batch(
                        model,
                        query_token_map[qid],
                        pid_batch,
                        passage_token_getter,
                        EVAL_BATCH_SIZE,
                    )
                )
            reranked = sorted(zip(candidate_pids, scores), key=lambda item: item[1], reverse=True)
            reranked_pids = [pid for pid, _ in reranked]
            bm25_ranked = sorted(zip(candidate_pids, candidate_bm25_ranks), key=lambda item: item[1])
            bm25_ranked_pids = [pid for pid, _ in bm25_ranked]
            relevant_pids = qrels[qid]
            bm25_metrics = ranking_metrics_at_10(bm25_ranked_pids, relevant_pids)
            vanilla_metrics = ranking_metrics_at_10(reranked_pids, relevant_pids)
            for metric_name, metric_value in bm25_metrics.items():
                bm25_metric_values[metric_name].append(metric_value)
            for metric_name, metric_value in vanilla_metrics.items():
                vanilla_metric_values[metric_name].append(metric_value)
            evaluated_queries += 1
            for rank, (pid, score) in enumerate(reranked, start=1):
                run_handle.write(f"{qid} Q0 {pid} {rank} {score:.6f} VANILLA_TRANSFORMER" + chr(10))

    if was_training:
        model.train()
    metrics = {
        "queries_evaluated": evaluated_queries,
    }
    for metric_name in RANKING_METRIC_NAMES:
        values = bm25_metric_values[metric_name]
        metrics[f"bm25_{metric_name}"] = float(np.mean(values)) if values else 0.0
    for metric_name in RANKING_METRIC_NAMES:
        values = vanilla_metric_values[metric_name]
        metrics[f"vanilla_{metric_name}"] = float(np.mean(values)) if values else 0.0
    metrics["run_path"] = str(run_path)
    print(metrics)
    return metrics


In [ ]:
TRAIN_LOG_FIELDNAMES = [
    "time",
    "epoch",
    "batch_idx",
    "global_step",
    "loss",
    "loss_ema",
    "pairwise_acc",
    "margin",
    "pos_score",
    "neg_score",
    "lr",
    "grad_norm",
]
DEV_METRICS_FIELDNAMES = [
    "epoch",
    "global_step",
    "dev_eval_mode",
    "queries_evaluated",
    "bm25_mrr@10",
    "bm25_hit@1",
    "bm25_hit@3",
    "bm25_hit@5",
    "bm25_hit@10",
    "bm25_ndcg@10",
    "vanilla_mrr@10",
    "vanilla_hit@1",
    "vanilla_hit@3",
    "vanilla_hit@5",
    "vanilla_hit@10",
    "vanilla_ndcg@10",
    "run_path",
    "metrics_path",
]


def build_dev_eval_log_payload(metrics, dev_eval_mode: str, epoch: Optional[int] = None, global_step: Optional[int] = None, checkpoint_path: Optional[Path] = None):
    payload = {}
    if epoch is not None:
        payload["epoch"] = int(epoch)
    if global_step is not None:
        payload["global_step"] = int(global_step)
    payload["dev_eval_mode"] = dev_eval_mode
    payload["queries_evaluated"] = int(metrics["queries_evaluated"])
    for prefix in ("vanilla", "bm25"):
        for metric_name in RANKING_METRIC_NAMES:
            compact_name = metric_name.replace("@", "")
            payload[f"dev_eval/{prefix}_{compact_name}"] = round(float(metrics[f"{prefix}_{metric_name}"]), 6)
    if checkpoint_path is not None:
        payload["checkpoint_path"] = str(checkpoint_path)
    return payload


def save_json(path: Path, payload) -> None:
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def create_train_log_writer(path: Path):
    handle = path.open("w", encoding="utf-8", newline="")
    writer = csv.DictWriter(handle, fieldnames=TRAIN_LOG_FIELDNAMES)
    writer.writeheader()
    writer._handle = handle
    return writer


def close_train_log_writer(writer) -> None:
    if writer is not None and hasattr(writer, "_handle"):
        writer._handle.close()


def save_checkpoint(
    path,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    config,
    metrics,
):
    raw_model = unwrap_model(model)
    torch.save(
        {
            "model_state_dict": raw_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict() if scheduler is not None else None,
            "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
            "epoch": epoch + 1,
            "global_step": global_step,
            "best_metric": metrics.get("best_dev_mrr10"),
            "best_checkpoint_path": metrics.get("best_dev_mrr10_checkpoint_path"),
            "config": config,
            "metrics": metrics,
        },
        path,
    )


def load_model_weights_for_eval(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    unwrap_model(model).load_state_dict(checkpoint["model_state_dict"])
    return model


def restore_training_state(model, optimizer, scheduler, scaler, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    unwrap_model(model).load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if scheduler is not None and checkpoint.get("scheduler_state_dict") is not None:
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    if scaler is not None and checkpoint.get("scaler_state_dict") is not None:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    return checkpoint


def save_csv_rows(path: Path, fieldnames, rows) -> None:
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def build_epoch_eval_paths(epoch: int, mode_label: str) -> Tuple[Path, Path, Path]:
    epoch_name = f"dev_epoch_{epoch + 1:03d}_{mode_label}"
    run_path = EPOCH_EVAL_DIR / f"{epoch_name}.run"
    metrics_path = EPOCH_EVAL_DIR / f"{epoch_name}_metrics.json"
    summary_path = EPOCH_EVAL_DIR / f"{epoch_name}_summary.json"
    return run_path, metrics_path, summary_path


def run_epoch_dev_eval(model, epoch: int, global_step: int):
    del model, epoch, global_step
    return None, "final_only"


def select_final_eval_checkpoint_path() -> Optional[Path]:
    checkpoint_candidates = [
        best_train_loss_checkpoint_path,
        BEST_TRAIN_LOSS_CHECKPOINT_PATH,
        LAST_CHECKPOINT_PATH,
    ]
    for checkpoint_path in checkpoint_candidates:
        if checkpoint_path is None:
            continue
        checkpoint_path = Path(checkpoint_path)
        if checkpoint_path.exists():
            return checkpoint_path
    return None


def run_final_full_dev_eval(model, device: torch.device):
    checkpoint_path = select_final_eval_checkpoint_path()
    if checkpoint_path is None:
        raise FileNotFoundError("No checkpoint is available for final full dev evaluation.")

    metrics = None
    if is_main_process():
        eval_model = unwrap_model(model)
        load_model_weights_for_eval(eval_model, checkpoint_path, device)
        with torch.no_grad():
            metrics = evaluate_reranker(
                eval_model,
                final_dev_candidates,
                final_dev_qrels,
                dev_query_tokens,
                run_path=FINAL_RUN_PATH,
                passage_token_getter=get_cached_passage_tokens,
                query_limit=None,
            )
        metrics["dev_eval_mode"] = "final_full"
        metrics["checkpoint_path"] = str(checkpoint_path)
        metrics["metrics_path"] = str(FINAL_METRICS_PATH)
        save_json(FINAL_METRICS_PATH, metrics)
        print(build_dev_eval_log_payload(metrics, "final_full", checkpoint_path=checkpoint_path))
    ddp_barrier()
    return metrics, checkpoint_path


def train_one_epoch(
    model,
    train_loader,
    optimizer,
    scheduler,
    device,
    epoch,
    global_step,
    max_train_steps,
    grad_accum_steps,
    max_grad_norm,
    scaler=None,
    precision="32-true",
    log_writer=None,
    loss_ema=None,
):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    progress_bar = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        disable=not is_main_process(),
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
    )
    micro_accumulator = {key: [] for key in ["loss", "pairwise_acc", "margin", "pos_score", "neg_score"]}
    step_rows = []
    stop_training = False

    for batch_idx, (pos_batch, neg_batch) in progress_bar:
        pos_batch = move_batch_to_device(pos_batch, device)
        neg_batch = move_batch_to_device(neg_batch, device)
        is_last_batch = batch_idx + 1 == len(train_loader)
        should_step = ((batch_idx + 1) % grad_accum_steps == 0) or is_last_batch
        sync_context = nullcontext()
        if is_dist_available_and_initialized() and hasattr(model, "no_sync") and not should_step:
            sync_context = model.no_sync()

        with sync_context:
            with get_autocast_context(device, precision):
                batch_metrics = compute_pairwise_batch_metrics(model, pos_batch, neg_batch)
                loss = batch_metrics["loss"]
                loss_for_backward = loss / grad_accum_steps
            if scaler is not None:
                scaler.scale(loss_for_backward).backward()
            else:
                loss_for_backward.backward()

        for key in micro_accumulator:
            micro_accumulator[key].append(batch_metrics[key].detach())

        if not should_step:
            continue

        if scaler is not None:
            scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(unwrap_model(model).parameters(), max_grad_norm)
        if scaler is not None:
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        if scheduler is not None:
            scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

        reduced_metrics = {}
        for key, values in micro_accumulator.items():
            stacked = torch.stack(values)
            reduced_metrics[key] = float(reduce_mean(stacked.mean()).item())
        grad_norm_tensor = torch.as_tensor(float(grad_norm), device=device)
        grad_norm_value = float(reduce_mean(grad_norm_tensor).item())
        lr_tensor = torch.as_tensor(float(optimizer.param_groups[0]["lr"]), device=device)
        lr_value = float(reduce_mean(lr_tensor).item())
        loss_ema = reduced_metrics["loss"] if loss_ema is None else (0.98 * loss_ema + 0.02 * reduced_metrics["loss"])

        row = {
            "time": time.time(),
            "epoch": epoch + 1,
            "batch_idx": batch_idx,
            "global_step": global_step,
            "loss": reduced_metrics["loss"],
            "loss_ema": loss_ema,
            "pairwise_acc": reduced_metrics["pairwise_acc"],
            "margin": reduced_metrics["margin"],
            "pos_score": reduced_metrics["pos_score"],
            "neg_score": reduced_metrics["neg_score"],
            "lr": lr_value,
            "grad_norm": grad_norm_value,
        }
        step_rows.append(row)

        if log_writer is not None and is_main_process():
            log_writer.writerow(row)
            log_writer._handle.flush()

        if is_main_process():
            progress_bar.set_postfix(
                {
                    "global_step": global_step,
                    "loss": f"{row['loss']:.4f}",
                    "loss_ema": f"{row['loss_ema']:.4f}",
                    "acc": f"{row['pairwise_acc']:.3f}",
                    "margin": f"{row['margin']:.3f}",
                    "lr": f"{row['lr']:.2e}",
                    "grad_norm": f"{row['grad_norm']:.3f}",
                }
            )

        micro_accumulator = {key: [] for key in micro_accumulator}
        if max_train_steps is not None and global_step >= int(max_train_steps):
            stop_training = True
            break

    if is_main_process():
        progress_bar.close()

    if step_rows:
        epoch_summary = {
            "epoch": epoch + 1,
            "global_step": global_step,
            "optimizer_steps": len(step_rows),
            "loss": float(np.mean([row["loss"] for row in step_rows])),
            "loss_ema": float(step_rows[-1]["loss_ema"]),
            "pairwise_acc": float(np.mean([row["pairwise_acc"] for row in step_rows])),
            "margin": float(np.mean([row["margin"] for row in step_rows])),
            "pos_score": float(np.mean([row["pos_score"] for row in step_rows])),
            "neg_score": float(np.mean([row["neg_score"] for row in step_rows])),
            "lr": float(step_rows[-1]["lr"]),
            "grad_norm": float(np.mean([row["grad_norm"] for row in step_rows])),
        }
    else:
        epoch_summary = {
            "epoch": epoch + 1,
            "global_step": global_step,
            "optimizer_steps": 0,
            "loss": None,
            "loss_ema": loss_ema,
            "pairwise_acc": None,
            "margin": None,
            "pos_score": None,
            "neg_score": None,
            "lr": float(optimizer.param_groups[0]["lr"]),
            "grad_norm": None,
        }
    return epoch_summary, global_step, stop_training, loss_ema


def count_model_parameters(model) -> int:
    return sum(parameter.numel() for parameter in unwrap_model(model).parameters())


def count_trainable_parameters(model) -> int:
    return sum(parameter.numel() for parameter in unwrap_model(model).parameters() if parameter.requires_grad)


def get_trainable_parameters(model):
    return [parameter for parameter in unwrap_model(model).parameters() if parameter.requires_grad]


def resolve_token_embedding_parameter(model):
    raw_model = unwrap_model(model)
    if not hasattr(raw_model, "inner") or not hasattr(raw_model.inner, "embed_tokens"):
        raise AttributeError("Could not locate token embedding layer at model.inner.embed_tokens")
    token_embedding_layer = raw_model.inner.embed_tokens
    if not hasattr(token_embedding_layer, "embedding_weight"):
        raise AttributeError("Token embedding layer does not expose embedding_weight")
    return token_embedding_layer.embedding_weight, "inner.embed_tokens.embedding_weight"


def resolve_segment_embedding_parameter(model):
    raw_model = unwrap_model(model)
    if not hasattr(raw_model, "inner") or not hasattr(raw_model.inner, "segment_emb"):
        raise AttributeError("Could not locate segment embedding layer at model.inner.segment_emb")
    segment_embedding_layer = raw_model.inner.segment_emb
    if not hasattr(segment_embedding_layer, "embedding_weight"):
        raise AttributeError("Segment embedding layer does not expose embedding_weight")
    return segment_embedding_layer.embedding_weight, "inner.segment_emb.embedding_weight"


def apply_token_embedding_freeze(model):
    token_embedding_parameter, token_embedding_path = resolve_token_embedding_parameter(model)
    if FREEZE_TOKEN_EMBEDDINGS:
        token_embedding_parameter.requires_grad_(False)

    segment_embedding_parameter, segment_embedding_path = resolve_segment_embedding_parameter(model)
    if FREEZE_SEGMENT_EMBEDDINGS:
        raise ValueError("This ablation notebook must not freeze segment embeddings.")
    if not segment_embedding_parameter.requires_grad:
        raise RuntimeError("Segment embeddings unexpectedly became frozen in the ablation notebook.")

    return {
        "token_embedding_layer_path": token_embedding_path,
        "segment_embedding_layer_path": segment_embedding_path,
        "token_embeddings_frozen": not bool(token_embedding_parameter.requires_grad),
        "segment_embeddings_frozen": not bool(segment_embedding_parameter.requires_grad),
    }


device = setup_distributed()
WORLD_SIZE = get_world_size()
model = VanillaTransformerReranker(model_config).to(device)
total_parameters = count_model_parameters(model)
trainable_parameters_before_freeze = count_trainable_parameters(model)
freeze_diagnostics = apply_token_embedding_freeze(model)
trainable_parameters_after_freeze = count_trainable_parameters(model)
if SHOULD_USE_DDP:
    model = DDP(
        model,
        device_ids=[get_local_rank()],
        output_device=get_local_rank(),
        find_unused_parameters=False,
    )

optimizer_parameters = get_trainable_parameters(model)
optimizer_parameter_count = sum(parameter.numel() for parameter in optimizer_parameters)
if optimizer_parameter_count != trainable_parameters_after_freeze:
    raise RuntimeError("Optimizer parameter list does not match the post-freeze trainable parameter count.")

ablation_diagnostics = {
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "model_family": "vanilla_transformer_reranker",
    "baseline_type": "parameter_matched_non_recursive_transformer",
    "num_transformer_layers": VANILLA_TRANSFORMER_LAYERS,
    "trm_specific_components_removed": True,
    "model_family": "vanilla_transformer_reranker",
    "baseline_type": "parameter_matched_non_recursive_transformer",
    "num_transformer_layers": VANILLA_TRANSFORMER_LAYERS,
    "trm_specific_components_removed": True,
    "token_embeddings_frozen": freeze_diagnostics["token_embeddings_frozen"],
    "segment_embeddings_frozen": freeze_diagnostics["segment_embeddings_frozen"],
    "trainable_parameters_before_freeze": trainable_parameters_before_freeze,
    "trainable_parameters_after_freeze": trainable_parameters_after_freeze,
    "total_parameters": total_parameters,
    "token_embedding_layer_path": freeze_diagnostics["token_embedding_layer_path"],
}

optimizer = torch.optim.AdamW(
    optimizer_parameters,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=build_linear_warmup_decay_lambda(WARMUP_STEPS, ESTIMATED_TOTAL_STEPS),
)
scaler = torch.cuda.amp.GradScaler(enabled=device.type == "cuda" and EFFECTIVE_PRECISION == "16-mixed")

training_summary = {
    "profile": RUN_PROFILE,
    "num_train_examples": len(train_dataset),
    "epochs": EPOCHS,
    "max_train_steps": MAX_TRAIN_STEPS,
    "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
    "devices_requested": DEVICES,
    "world_size": WORLD_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "global_batch_size": GLOBAL_BATCH_SIZE,
    "estimated_steps_per_epoch": ESTIMATED_STEPS_PER_EPOCH,
    "estimated_total_steps": ESTIMATED_TOTAL_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_steps": WARMUP_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "precision": EFFECTIVE_PRECISION,
    "run_dir": str(RUN_DIR),
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "resume_checkpoint_path": RESUME_CHECKPOINT_PATH or None,
    "ddp_enabled": SHOULD_USE_DDP,
    "dev_eval_mode": "final_full_only",
    "dev_eval_fraction": None,
    "dev_eval_seed": None,
    "epoch_dev_eval_enabled": False,
    "final_dev_eval_only": True,
    "run_final_full_dev": RUN_FINAL_FULL_DEV,
    "train_triples_sample": TRAIN_TRIPLES_SAMPLE,
    "prep_manifest_path": str(PREP_MANIFEST_PATH),
    "run_data_manifest_path": str(RUN_DATA_MANIFEST_PATH),
    "sampled_train_triples_path": str(sampled_train_triples_path),
    "sampled_train_triples": sampled_train_triples_count,
    "sampled_train_queries": sampled_train_queries_count,
    "sampled_train_passages": sampled_train_passages_count,
    "epoch_dev_queries": epoch_dev_queries_count,
    "epoch_dev_candidate_rows": epoch_dev_candidate_rows_count,
    "ablation_notebook": IS_ABLATION_NOTEBOOK,
    "ablation_variant": ABLATION_OUTPUT_DIRNAME,
    "freeze_token_embeddings": FREEZE_TOKEN_EMBEDDINGS,
    "freeze_segment_embeddings": FREEZE_SEGMENT_EMBEDDINGS,
    "freeze_positional_components": FREEZE_POSITIONAL_COMPONENTS,
    "token_embedding_layer_path": freeze_diagnostics["token_embedding_layer_path"],
    "segment_embedding_layer_path": freeze_diagnostics["segment_embedding_layer_path"],
    "trainable_parameters_before_freeze": trainable_parameters_before_freeze,
    "trainable_parameters_after_freeze": trainable_parameters_after_freeze,
    "total_parameters": total_parameters,
}
if is_main_process():
    print(json.dumps(ablation_diagnostics, indent=2))
    save_json(TRAINING_CONFIG_PATH, training_summary)
    save_json(RUN_ARTIFACTS_PATH, run_artifacts)
    print(f"Checkpoint directory: {CHECKPOINT_DIR}")
    print(json.dumps(training_summary, indent=2))

start_epoch = 0
global_step = 0
best_train_loss = float("inf")
best_dev_mrr10 = None
best_train_loss_checkpoint_path = None
best_dev_mrr10_checkpoint_path = None
epoch_summaries = []
loss_ema = None
final_dev_metrics = None
final_eval_checkpoint_path = None

if RESUME_FROM_CHECKPOINT:
    resume_path = Path(RESUME_CHECKPOINT_PATH).expanduser() if RESUME_CHECKPOINT_PATH else LAST_CHECKPOINT_PATH
    if not resume_path.is_file():
        raise FileNotFoundError(f"Resume checkpoint not found: {resume_path}")
    if is_main_process():
        print(f"Loading resume checkpoint: {resume_path}")
    resume_checkpoint = restore_training_state(model, optimizer, scheduler, scaler if scaler.is_enabled() else None, resume_path, device)
    start_epoch = int(resume_checkpoint["epoch"])
    global_step = int(resume_checkpoint["global_step"])
    resume_metrics = resume_checkpoint.get("metrics") or {}
    best_train_loss = float(resume_metrics.get("best_train_loss", best_train_loss))
    best_dev_mrr10 = resume_metrics.get("best_dev_mrr10")
    best_train_loss_checkpoint_path = resume_metrics.get("best_train_loss_checkpoint_path")
    best_dev_mrr10_checkpoint_path = resume_metrics.get("best_dev_mrr10_checkpoint_path")
    loss_ema = resume_metrics.get("loss_ema")
    if is_main_process():
        print(f"Resuming training from epoch {start_epoch + 1}")
        print(f"Resumed global_step: {global_step}")
ddp_barrier()

train_log_writer = create_train_log_writer(TRAIN_LOG_PATH) if is_main_process() else None
stop_training = False

try:
    for epoch in range(start_epoch, EPOCHS):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)

        epoch_summary, global_step, stop_training, loss_ema = train_one_epoch(
            model=model,
            train_loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            epoch=epoch,
            global_step=global_step,
            max_train_steps=MAX_TRAIN_STEPS,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            max_grad_norm=MAX_GRAD_NORM,
            scaler=scaler if scaler.is_enabled() else None,
            precision=EFFECTIVE_PRECISION,
            log_writer=train_log_writer,
            loss_ema=loss_ema,
        )
        epoch_summary["loss_ema"] = loss_ema
        epoch_summary["dev_eval_mode"] = "final_only"
        epoch_summaries.append(epoch_summary)

        improved_train_loss = epoch_summary["loss"] is not None and epoch_summary["loss"] < best_train_loss
        if improved_train_loss:
            best_train_loss = float(epoch_summary["loss"])
            best_train_loss_checkpoint_path = str(BEST_TRAIN_LOSS_CHECKPOINT_PATH)

        if is_main_process():
            save_json(EPOCH_SUMMARIES_PATH, epoch_summaries)
            checkpoint_metrics = {
                "epoch_summary": epoch_summary,
                "best_train_loss": best_train_loss,
                "best_dev_mrr10": best_dev_mrr10,
                "best_train_loss_checkpoint_path": best_train_loss_checkpoint_path,
                "best_dev_mrr10_checkpoint_path": best_dev_mrr10_checkpoint_path,
                "loss_ema": loss_ema,
                "dev_eval_metrics": None,
                "dev_eval_mode": "final_only",
            }
            epoch_checkpoint_path = CHECKPOINT_DIR / f"epoch_{epoch + 1:03d}.pt"
            save_checkpoint(
                epoch_checkpoint_path,
                model,
                optimizer,
                scheduler,
                scaler if scaler.is_enabled() else None,
                epoch,
                global_step,
                training_summary,
                checkpoint_metrics,
            )
            save_checkpoint(
                LAST_CHECKPOINT_PATH,
                model,
                optimizer,
                scheduler,
                scaler if scaler.is_enabled() else None,
                epoch,
                global_step,
                training_summary,
                checkpoint_metrics,
            )
            if improved_train_loss:
                save_checkpoint(
                    BEST_TRAIN_LOSS_CHECKPOINT_PATH,
                    model,
                    optimizer,
                    scheduler,
                    scaler if scaler.is_enabled() else None,
                    epoch,
                    global_step,
                    training_summary,
                    checkpoint_metrics,
                )
            print(f"Saved epoch checkpoint: {epoch_checkpoint_path}")
            print(f"Updated last checkpoint: {LAST_CHECKPOINT_PATH}")
        ddp_barrier()
        if stop_training:
            break
finally:
    close_train_log_writer(train_log_writer)

if RUN_FINAL_FULL_DEV:
    final_dev_metrics, final_eval_checkpoint_path = run_final_full_dev_eval(model, device)

fit_summary = {
    "global_step": int(global_step),
    "epochs_completed": len(epoch_summaries),
    "checkpoint_dir": str(CHECKPOINT_DIR),
    "log_dir": str(LOG_DIR),
    "eval_dir": str(EVAL_DIR),
    "epoch_eval_dir": str(EPOCH_EVAL_DIR),
    "final_eval_dir": str(FINAL_EVAL_DIR),
    "run_artifacts_path": str(RUN_ARTIFACTS_PATH),
    "train_log_path": str(TRAIN_LOG_PATH),
    "dev_metrics_by_epoch_path": None,
    "last_checkpoint_path": str(LAST_CHECKPOINT_PATH) if LAST_CHECKPOINT_PATH.exists() else None,
    "best_train_loss_checkpoint_path": str(BEST_TRAIN_LOSS_CHECKPOINT_PATH) if BEST_TRAIN_LOSS_CHECKPOINT_PATH.exists() else None,
    "best_dev_mrr10_checkpoint_path": None,
    "final_eval_checkpoint_path": str(final_eval_checkpoint_path) if final_eval_checkpoint_path is not None else None,
    "final_metrics_path": str(FINAL_METRICS_PATH) if FINAL_METRICS_PATH.exists() else None,
    "dev_eval_mode": "final_full_only",
    "dev_eval_fraction": None,
    "epoch_dev_eval_enabled": False,
    "final_dev_eval_only": True,
}
if final_dev_metrics is not None:
    fit_summary["final_full_dev_mrr10"] = float(final_dev_metrics["vanilla_mrr@10"])
    fit_summary["final_full_dev_bm25_mrr10"] = float(final_dev_metrics["bm25_mrr@10"])
    fit_summary["final_full_dev_queries_evaluated"] = int(final_dev_metrics["queries_evaluated"])
if is_main_process():
    save_json(FIT_SUMMARY_PATH, fit_summary)
    print(json.dumps(fit_summary, indent=2))
fit_summary
